In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
from bkt.predict import BKTPredictor

# Load training data
data_path = Path('../data/processed/bkt_training_data.csv')
df = pd.read_csv(data_path)

print(f"Loaded {len(df):,} interactions, {df['user_id'].nunique():,} students, {df['skill_name'].nunique()} skills")

# Sort by user and chronological order — important for BKT
df = df.sort_values(['user_id', 'order_id']).reset_index(drop=True)

predictor = BKTPredictor.load()
print(f"Predictor loaded with {len(predictor.params)} skills")

Loaded 445,009 interactions, 3,669 students, 95 skills
Predictor loaded with 95 skills


In [2]:
# For each (student, skill) pair, compute final BKT mastery using their attempt sequence

mastery_matrix = defaultdict(dict)  # mastery_matrix[student_id][skill] = final P(known)

# Group attempts by (student, skill)
grouped = df.groupby(['user_id', 'skill_name'])['correct'].apply(list)

print(f"Processing {len(grouped):,} (student, skill) pairs...")

for (student_id, skill), attempts in grouped.items():
    if not predictor.has_skill(skill):
        continue
    final_mastery = predictor.predict(skill, attempts)
    mastery_matrix[student_id][skill] = final_mastery

# Convert to a DataFrame for easier analysis
mastery_df = pd.DataFrame.from_dict(mastery_matrix, orient='index')
print(f"\nMastery matrix shape: {mastery_df.shape}")
print(f"  Students: {mastery_df.shape[0]:,}")
print(f"  Skills:   {mastery_df.shape[1]}")
print(f"  Total non-null entries: {mastery_df.notna().sum().sum():,}")
print(f"  Density: {mastery_df.notna().sum().sum() / (mastery_df.shape[0] * mastery_df.shape[1]):.1%}")

Processing 38,273 (student, skill) pairs...

Mastery matrix shape: (3669, 95)
  Students: 3,669
  Skills:   95
  Total non-null entries: 38,273
  Density: 11.0%


In [3]:
# For each pair of skills, count how many students attempted both
skills = mastery_df.columns.tolist()
n_skills = len(skills)

print(f"Checking pairwise student overlap for {n_skills} skills...")

# Build an indicator matrix: 1 if student attempted skill, 0 otherwise
attempted = mastery_df.notna().astype(int)

# Co-attempt matrix: number of students who attempted both skill_i and skill_j
coattempt = attempted.T.dot(attempted)  # 95x95 matrix

# Look at distribution of overlaps
upper_triangle = coattempt.values[np.triu_indices(n_skills, k=1)]
print(f"\nNumber of skill pairs: {len(upper_triangle):,}")
print(f"Pair-wise student overlap statistics:")
print(f"  Min:    {upper_triangle.min():,}")
print(f"  Max:    {upper_triangle.max():,}")
print(f"  Median: {int(np.median(upper_triangle)):,}")
print(f"  Mean:   {upper_triangle.mean():.0f}")

# How many pairs have enough students for reliable correlation?
for threshold in [10, 30, 50, 100]:
    n_reliable = (upper_triangle >= threshold).sum()
    pct = n_reliable / len(upper_triangle) * 100
    print(f"\nPairs with ≥ {threshold} co-attempting students: {n_reliable:,} ({pct:.1f}%)")

Checking pairwise student overlap for 95 skills...

Number of skill pairs: 4,465
Pair-wise student overlap statistics:
  Min:    0
  Max:    919
  Median: 117
  Mean:   135

Pairs with ≥ 10 co-attempting students: 3,550 (79.5%)

Pairs with ≥ 30 co-attempting students: 3,251 (72.8%)

Pairs with ≥ 50 co-attempting students: 3,017 (67.6%)

Pairs with ≥ 100 co-attempting students: 2,427 (54.4%)


In [4]:
from scipy.stats import pearsonr

MIN_OVERLAP = 30
SIMILARITY_THRESHOLD = 0.3  # below this, treat as unrelated (set to 0)

print(f"Computing similarity matrix ({n_skills}x{n_skills} = {n_skills**2:,} entries)...")
print(f"  Minimum overlap for reliable correlation: {MIN_OVERLAP} students")
print(f"  Similarities below {SIMILARITY_THRESHOLD} clamped to 0")

similarity = np.zeros((n_skills, n_skills))

for i, skill_a in enumerate(skills):
    for j, skill_b in enumerate(skills):
        if i == j:
            similarity[i, j] = 1.0  # self-similarity
            continue
        if i > j:
            continue  # we'll mirror later, only compute upper triangle
        
        # Find students who attempted both
        both = mastery_df[[skill_a, skill_b]].dropna()
        
        if len(both) < MIN_OVERLAP:
            sim = 0.0
        else:
            r, _ = pearsonr(both[skill_a], both[skill_b])
            # Clamp negative correlations to 0 (noise) and apply threshold
            sim = max(0.0, r) if r >= SIMILARITY_THRESHOLD else 0.0
        
        similarity[i, j] = sim
        similarity[j, i] = sim  # symmetric

print(f"\nDone. Matrix shape: {similarity.shape}")

# Some quick stats
upper = similarity[np.triu_indices(n_skills, k=1)]
nonzero = upper[upper > 0]
print(f"\nNon-zero similarity pairs: {len(nonzero):,} / {len(upper):,} ({len(nonzero)/len(upper):.1%})")
if len(nonzero) > 0:
    print(f"  Mean similarity (non-zero pairs): {nonzero.mean():.3f}")
    print(f"  Max similarity:                   {nonzero.max():.3f}")
    print(f"  Median:                           {np.median(nonzero):.3f}")

Computing similarity matrix (95x95 = 9,025 entries)...
  Minimum overlap for reliable correlation: 30 students
  Similarities below 0.3 clamped to 0

Done. Matrix shape: (95, 95)

Non-zero similarity pairs: 872 / 4,465 (19.5%)
  Mean similarity (non-zero pairs): 0.425
  Max similarity:                   0.993
  Median:                           0.392


In [5]:
# Get all pairs with their similarity, sorted descending
pairs = []
for i in range(n_skills):
    for j in range(i + 1, n_skills):
        if similarity[i, j] > 0:
            pairs.append((skills[i], skills[j], similarity[i, j]))

pairs.sort(key=lambda x: -x[2])

print("Top 20 most similar skill pairs:")
print("-" * 70)
for skill_a, skill_b, sim in pairs[:20]:
    print(f"  {sim:.3f}  {skill_a:<35} ↔  {skill_b}")

print("\n\nBottom 10 of non-zero similarities (just above threshold):")
print("-" * 70)
for skill_a, skill_b, sim in pairs[-10:]:
    print(f"  {sim:.3f}  {skill_a:<35} ↔  {skill_b}")

Top 20 most similar skill pairs:
----------------------------------------------------------------------
  0.993  Range                               ↔  Write Linear Equation from Situation
  0.981  Multiplication and Division Positive Decimals ↔  Write Linear Equation from Situation
  0.978  Number Line                         ↔  Solving Inequalities
  0.961  Pattern Finding                     ↔  Write Linear Equation from Situation
  0.953  Volume Sphere                       ↔  Angles on Parallel Lines Cut by a Transversal
  0.892  Least Common Multiple               ↔  Area Rectangle
  0.871  Table                               ↔  Fraction Of
  0.867  Probability of a Single Event       ↔  Write Linear Equation from Situation
  0.861  Subtraction Whole Numbers           ↔  Write Linear Equation from Situation
  0.845  Order of Operations +,-,/,* () positive reals ↔  Probability of a Single Event
  0.817  Circle Graph                        ↔  Equivalent Fractions
  0.807  Interior 

In [6]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# all-MiniLM-L6-v2 is small (80MB), fast, and good for short text similarity
print("Loading sentence transformer model...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# Embed all skill names
skill_names = list(predictor.params.keys())
print(f"Embedding {len(skill_names)} skill names...")
embeddings = embed_model.encode(skill_names, show_progress_bar=True)
print(f"Embeddings shape: {embeddings.shape}")  # (95, 384)

# Compute cosine similarity matrix
similarity = cosine_similarity(embeddings)
print(f"Similarity matrix shape: {similarity.shape}")

# Stats
n_skills = len(skill_names)
upper = similarity[np.triu_indices(n_skills, k=1)]
print(f"\nSimilarity statistics:")
print(f"  Min:    {upper.min():.3f}")
print(f"  Max:    {upper.max():.3f}")
print(f"  Mean:   {upper.mean():.3f}")
print(f"  Median: {np.median(upper):.3f}")

c:\Users\Lenovo\Documents\Research\meta_agent_pro\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading sentence transformer model...


c:\Users\Lenovo\Documents\Research\meta_agent_pro\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8541.36it/s]


Embedding 95 skill names...


Batches: 100%|██████████| 3/3 [00:00<00:00, 14.19it/s]

Embeddings shape: (95, 384)
Similarity matrix shape: (95, 95)

Similarity statistics:
  Min:    -0.161
  Max:    0.966
  Mean:   0.168
  Median: 0.145


In [7]:
# Get all pairs with their similarity, sorted descending
pairs = []
for i in range(n_skills):
    for j in range(i + 1, n_skills):
        pairs.append((skill_names[i], skill_names[j], similarity[i, j]))

pairs.sort(key=lambda x: -x[2])

print("Top 20 most semantically similar skill pairs:")
print("-" * 75)
for skill_a, skill_b, sim in pairs[:20]:
    print(f"  {sim:.3f}  {skill_a:<38} ↔  {skill_b}")

Top 20 most semantically similar skill pairs:
---------------------------------------------------------------------------
  0.966  Equation Solving Two or Fewer Steps    ↔  Equation Solving More Than Two Steps
  0.846  Solving Systems of Linear Equations by Graphing ↔  Solving Systems of Linear Equations
  0.843  Percent Of                             ↔  Percents
  0.819  Multiplication Fractions               ↔  Division Fractions
  0.817  Finding Percents                       ↔  Percents
  0.807  Probability of a Single Event          ↔  Probability of Two Distinct Events
  0.799  Surface Area Rectangular Prism         ↔  Volume Rectangular Prism
  0.784  Addition and Subtraction Integers      ↔  Addition and Subtraction Fractions
  0.773  Addition and Subtraction Integers      ↔  Addition Whole Numbers
  0.763  Multiplication and Division Integers   ↔  Multiplication and Division Positive Decimals
  0.759  Fraction Of                            ↔  Division Fractions
  0.758  Additi

In [8]:
import json
from pathlib import Path

SIMILARITY_THRESHOLD = 0.5

# Build the dictionary structure: {skill: {related_skill: similarity}}
similarity_dict = {}
for i, skill in enumerate(skill_names):
    related = {}
    for j, other_skill in enumerate(skill_names):
        if i == j:
            continue
        sim = float(similarity[i, j])
        if sim >= SIMILARITY_THRESHOLD:
            related[other_skill] = sim
    similarity_dict[skill] = related

# Stats
n_with_relations = sum(1 for r in similarity_dict.values() if r)
total_relations = sum(len(r) for r in similarity_dict.values()) // 2  # divide by 2 since symmetric
print(f"Skills with at least one relation: {n_with_relations} / {len(skill_names)}")
print(f"Total related-skill pairs: {total_relations}")
print(f"Average relations per skill: {total_relations * 2 / len(skill_names):.1f}")

# Save
output_path = Path('../models/skill_similarity.json')
with open(output_path, 'w') as f:
    json.dump(similarity_dict, f, indent=2)

print(f"\nSaved to {output_path}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")

Skills with at least one relation: 65 / 95
Total related-skill pairs: 141
Average relations per skill: 3.0

Saved to ..\models\skill_similarity.json
File size: 18.1 KB


In [3]:
import sys
sys.path.append('..')

import importlib
import bkt.predict
importlib.reload(bkt.predict)
from bkt.predict import BKTPredictor, ColdStartPriorCalculator

predictor = BKTPredictor.load()
cold_start = ColdStartPriorCalculator()

# Test 1: A student who's mastered Adding Fractions, encountering Subtracting Fractions
print("Test 1 — Student strong on related skill")
student_masteries = {"Addition and Subtraction Fractions": 0.95}
result = cold_start.compute_prior(
    skill="Equivalent Fractions",
    student_masteries=student_masteries,
    population_prior=predictor.params["Equivalent Fractions"]["prior"],
)
print(f"  Result: {result}")
print()

# Test 2: New student with no mastery anywhere
print("Test 2 — Brand new student, no prior data")
result = cold_start.compute_prior(
    skill="Equivalent Fractions",
    student_masteries={},
    population_prior=predictor.params["Equivalent Fractions"]["prior"],
)
print(f"  Result: {result}")
print()

# Test 3: A student weak on related skills
print("Test 3 — Student weak on related skills")
student_masteries = {"Addition and Subtraction Fractions": 0.05}
result = cold_start.compute_prior(
    skill="Equivalent Fractions",
    student_masteries=student_masteries,
    population_prior=predictor.params["Equivalent Fractions"]["prior"],
)
print(f"  Result: {result}")

Test 1 — Student strong on related skill
  Result: {'prior': 0.95, 'used_transfer': True, 'related_skills_used': ['Addition and Subtraction Fractions'], 'transfer_evidence': 'Population prior: 0.607, weighted related mastery: 0.950, using: 0.950'}

Test 2 — Brand new student, no prior data
  Result: {'prior': 0.6072801459104594, 'used_transfer': False, 'related_skills_used': [], 'transfer_evidence': 'No related skills attempted by student'}

Test 3 — Student weak on related skills
  Result: {'prior': 0.6072801459104594, 'used_transfer': False, 'related_skills_used': ['Addition and Subtraction Fractions'], 'transfer_evidence': 'Population prior: 0.607, weighted related mastery: 0.050, using: 0.607'}
